In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
train = pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/train.csv")
train.head(10)

In [ ]:
train.shape

In [ ]:
train.isnull().sum()

In [ ]:
train[train["sales"] > 0].describe()

In [ ]:
train.info()

In [ ]:
test = pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/test.csv")
test.head()

In [ ]:
sample_submission = pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/sample_submission.csv")
sample_submission.head(10)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

class DataProcessor:
    def __init__(self, train_full_df, test_full_df, drop_columns):
        self.train_full_df = train_full_df
        self.test_full_df = test_full_df
        self.drop_columns = drop_columns
        self.label_columns = self._describe_data()
        self.train_X_df, self.train_y, self.test_X_df = self._separate_values()
        self.train_X_preprocessed, self.test_X_preprocessed = self._preprocess()
        self.X_train, self.X_test, self.y_train, self.y_test, self.X_submit = self._split_data()
        

    def _describe_data(self):
        print(f"Number of rows in training: {len(train)}")
        print(f"Number of rows in test: {len(test)}")
    
        columns_train = set(self.train_full_df.columns)
        columns_test = set(self.test_full_df.columns)
    
        label_columns = list(columns_train - columns_test)
        return label_columns
        
    def _separate_values(self):
        train_y = self.train_full_df[self.label_columns].values
        train_X = self.train_full_df.drop(columns=self.drop_columns)
        train_X = train_X.drop(columns=self.label_columns)
        test_X = self.test_full_df.drop(columns=self.drop_columns)

        return train_X, train_y, test_X

    def _preprocess(self):
        # Identify numerical and categorical columns
        numerical_cols = self.train_X_df.select_dtypes(include=['int64', 'float64']).columns
        categorical_cols = self.train_X_df.select_dtypes(include=['object', 'category']).columns
        
        # 2. Define preprocessing steps using Scikit-learn's Pipeline and ColumnTransformer
        numerical_transformer = Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='mean')),
            ('scaler', StandardScaler())
        ])
        
        categorical_transformer = Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ])
        
        preprocessor = ColumnTransformer(
            transformers=[
                ('num', numerical_transformer, numerical_cols),
                ('cat', categorical_transformer, categorical_cols)
            ],
            remainder='passthrough' # Keep any other columns not specified (e.g., if you had ID columns)
        )
        
        train_X_preprocessed = preprocessor.fit_transform(self.train_X_df)
        test_X_preprocessed = preprocessor.transform(self.test_X_df)

        return train_X_preprocessed, test_X_preprocessed

    def _split_data(self):
        X_train, X_test, y_train, y_test = self.train_X_preprocessed[:3000000], self.train_X_preprocessed[3000000:], self.train_y[:3000000], self.train_y[3000000:]
        train_test_split(self.train_X_preprocessed, self.train_y, test_size=0.3, random_state=42)
    
        return X_train, X_test, y_train, y_test, self.test_X_preprocessed

In [ ]:
data_processor = DataProcessor(train, test, ["date"])

In [ ]:
data_processor.label_columns

In [ ]:
data_processor.X_train.shape

In [ ]:
data_processor.X_train[:1]

In [ ]:
data_processor.X_test.shape

In [ ]:
import torch
from torch import nn

### Create dataset and dataloaders

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class SalesDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32).unsqueeze(1)
        self.labels = torch.tensor(labels,  dtype=torch.float32)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

In [ ]:
train_dataset = SalesDataset(data_processor.X_train, data_processor.y_train)
test_dataset = SalesDataset(data_processor.X_test, data_processor.y_test)

In [ ]:
train_dataset.labels.shape

In [ ]:
train_dataset.features.shape

In [ ]:
BATCH_SIZE = 32

train_dataloader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE)
test_dataloader = DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE)

In [ ]:
features, label = next(iter(train_dataloader))
features.shape, label.shape

### Create model

In [ ]:
# Setup device agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
class LSTMPredictor(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super(LSTMPredictor, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        # Change from nn.RNN to nn.LSTM
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        # Initialize the hidden state (h0) and cell state (c0) with zeros
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)

        # Pass the input and both states through the LSTM layer
        # The output of LSTM is a tuple: (output, (h_n, c_n))
        out, (hn, cn) = self.lstm(x, (h0, c0))

        # Use the output from the last time step for the prediction
        out = self.fc(out[:, -1, :])

        out = self.relu(out)
        
        return out

In [ ]:
# Hyperparameters
input_size = 36
hidden_size = 20
num_layers = 1
num_epochs = 10
learning_rate = 0.01

# Instantiate the model
model = LSTMPredictor(input_size, hidden_size, num_layers).to(device)

# Loss and optimizer
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
features.shape

In [ ]:
model(features.to(device))

In [ ]:
# Create train_step()
def train_step(model: torch.nn.Module,
              dataloader: torch.utils.data.DataLoader,
              loss_fn: torch.nn.Module,
              optimizer: torch.optim.Optimizer,
              device=device):
    # Put the model in train mode
    model.train()

    # Setup train loss and train accuracy values
    train_loss, train_acc = 0, 0

    # Loop through data loade data batches
    for batch, (X, y) in enumerate(dataloader):
        # Send data to the target device
        X, y = X.to(device), y.to(device)

        # Forward pass
        y_logits = model(X)
        y_pred = y_logits
        
        # Calculate the loss
        loss = loss_fn(y_logits, y)
        train_loss += loss.item()

        # Optimizer zero grad
        optimizer.zero_grad()

        # Loss backward
        loss.backward()

        # Optimizer step
        optimizer.step()

        # calculate the accuracy metric
        # y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
        # print(y_pred_class.shape)
        log_pred = torch.log1p(y_pred) # log1p(x) is equivalent to log(x + 1)
        log_true = torch.log1p(y)
        
        # Calculate MSE on the log-transformed values
        squared_error = torch.mean((log_pred - log_true)**2)
        
        # Take the square root to get RMSLE
        train_acc += torch.sqrt(squared_error)

    # Adjust metrics to get the average loss and accuracy per batch
    train_loss = train_loss / len(dataloader)
    train_acc = train_acc / len(dataloader)
    return train_loss, train_acc

In [ ]:
# Create a test step
def test_step(
    model: torch.nn.Module,
    dataloader: torch.utils.data.DataLoader,
    loss_fn: torch.nn.Module,
    device=device):

    # Put the model in eval mode
    model.eval()

    # Setup test loss and test accuracy
    test_loss, test_acc = 0, 0

    # Turn on inference mode
    with torch.inference_mode():
        # Loop through DataLoader batches
        for batch, (X, y) in enumerate(dataloader):
            # Send data to target device
            X, y = X.to(device), y.to(device)

            # Forward pass
            test_pred_logits = model(X)
            test_preds = test_pred_logits

            # calculate the loss
            loss = loss_fn(test_pred_logits, y)
            test_loss += loss.item()

            log_pred = torch.log1p(test_preds) # log1p(x) is equivalent to log(x + 1)
            log_true = torch.log1p(y)
            
            # Calculate MSE on the log-transformed values
            squared_error = torch.mean((log_pred - log_true)**2)
            # calculate the accuracy
            # test_pred_labels = test_pred_logits.argmax(dim=1)
            test_acc += torch.sqrt(squared_error)

        # Adjust metrics to get average loss and accuracy per batch
        test_loss = test_loss / len(dataloader)
        test_acc = test_acc / len(dataloader)
        return test_loss, test_acc

In [ ]:
from tqdm.auto import tqdm

# Create a train function that takes in various model parameters
def train(model: torch.nn.Module,
         train_dataloader: torch.utils.data.DataLoader,
         test_dataloader: torch.utils.data.DataLoader,
         optimizer: torch.optim.Optimizer,
         loss_fn: torch.nn.Module = nn.CrossEntropyLoss(),
         epochs: int = 5,
         device=device):
    # Create a empty results dictionary
    results = {"train_loss" : [],
              "train_acc": [],
              "test_loss": [],
              "test_acc": []}

    # Loop through training and testing steps for a number of epochs
    for epoch in tqdm(range(epochs)):
        train_loss, train_acc = train_step(model=model,
                                          dataloader=train_dataloader,
                                          loss_fn=loss_fn,
                                          optimizer=optimizer,
                                          device=device)
        test_loss, test_acc = test_step(model=model,
                                       dataloader=test_dataloader,
                                       loss_fn=loss_fn,
                                       device=device)

        # Print out what is happening
        print(f"Epoch: {epoch} | Train loss: {train_loss:.4f}, Train acc: {test_acc: .4f} | Test loss : {test_loss:.4f}, Test acc: {test_acc:.4f}")

        # Update results dictionary
        results["train_loss"].append(train_loss)
        results["train_acc"].append(train_acc)
        results["test_loss"].append(test_loss)
        results["test_acc"].append(test_acc)

    # Return the filled results at the end of the epochs
    return results

In [ ]:
# Set the random seed
torch.manual_seed(42)
torch.cuda.manual_seed(42)

# Set number of epochs
NUM_EPOCHS = 5

# Start the timer
from timeit import default_timer as timer
start_time = timer()

# Train_model_0
model_results = train(model=model.to(device),
                       train_dataloader=train_dataloader,
                       test_dataloader=test_dataloader,
                       optimizer=optimizer,
                       loss_fn=loss_fn,
                       epochs=NUM_EPOCHS)

# End the timer and print out time
end_time = timer()

print(f"Total training time : {end_time-start_time: .3f} seconds")

In [ ]:
with torch.inference_mode():
    submit = model(torch.tensor(data_processor.X_submit, dtype=torch.float32).unsqueeze(1).to(device)).to(device)

predictions = submit
predictions

In [ ]:
predictions.shape

In [ ]:
output = pd.DataFrame({'id' : test["id"], 'sales': predictions.cpu().squeeze()})
output.to_csv('submission.csv', index=False)
print('Submission saved!')